# Stage 05: Data Storage

Stage 05. **I based it on the lecture notebook.**

In [1]:
# Install missing packages (uncomment and run to install).
# !pip install pandas pyarrow python-dotenv

In [2]:
from pathlib import Path

# Project root.
ROOT = Path.cwd()
if not (ROOT / ".env.example").exists() and (ROOT.parent / ".env.example").exists():
    ROOT = ROOT.parent

CHECKS = [
    (".env", "NEEDED", "copy .env.example to .env"),
    (".env.example", "NEEDED", "template for local secrets"),
    ("src/io_utils.py", "NEEDED", "save and load helpers"),
    ("data/raw/prismatic_evoluations_prices.csv", "NEEDED", "Stage 03 sample prices"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    raise FileNotFoundError(f"{missing} needed file(s) missing under {ROOT}")
print("\nAll needed files present.")

Looking in: /Users/ghostof0days/projects/bootcamp/project

  [OK ]  NEEDED    .env                                copy .env.example to .env
  [OK ]  NEEDED    .env.example                        template for local secrets
  [OK ]  NEEDED    src/io_utils.py                     save and load helpers
  [OK ]  NEEDED    data/raw/prismatic_evoluations_prices.csv  Stage 03 sample prices

All needed files present.


In [3]:
import sys
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

# Import helpers from src/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import get_key
from src.io_utils import read_df, validate_loaded, write_df

# Load `.env`.
load_dotenv(ROOT / ".env")

# Env paths.
RAW = ROOT / (get_key("DATA_DIR_RAW", "data/raw") or "data/raw")
PROC = ROOT / (get_key("DATA_DIR_PROCESSED", "data/processed") or "data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print("RAW ->", RAW.resolve())
print("PROC ->", PROC.resolve())

RAW -> /Users/ghostof0days/projects/bootcamp/project/data/raw
PROC -> /Users/ghostof0days/projects/bootcamp/project/data/processed


## Load sample prices

In [4]:
# Prismatic Evolutions sample.
card_prices = pd.read_csv(RAW / "prismatic_evoluations_prices.csv", parse_dates=["date"])
card_prices.head()

,card_name,rarity,market_price,date
0,Umbreon ex,Special Illustration Rare,139.20,2026-08-01
1,Sylveon ex,Special Illustration Rare,94.08,2026-08-01
2,Espeon ex,Special Illustration Rare,69.12,2026-08-01
3,Leafeon ex,Ultra Rare,17.76,2026-08-01
4,Eevee,Common,0.24,2026-08-01


## Save CSV and Parquet

In [5]:
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# Save CSV.
csv_path = RAW / f"sample_{stamp}.csv"
card_prices.to_csv(csv_path, index=False)
print("Saved CSV ->", csv_path)

# Save Parquet.
parquet_path = PROC / f"sample_{stamp}.parquet"
try:
    card_prices.to_parquet(parquet_path)
    print("Saved Parquet ->", parquet_path)
except Exception as error:
    print("Parquet engine not available. Install pyarrow or fastparquet.")
    print("Error:", error)
    parquet_path = None

Saved CSV -> /Users/ghostof0days/projects/bootcamp/project/data/raw/sample_20260817-053114.csv
Saved Parquet -> /Users/ghostof0days/projects/bootcamp/project/data/processed/sample_20260817-053114.parquet


## Reload and validate

In [6]:
# Reload CSV.
reloaded_csv = pd.read_csv(csv_path, parse_dates=["date"])
csv_check = validate_loaded(card_prices, reloaded_csv)
print("CSV validation:", csv_check)

if parquet_path is not None:
    try:
        # Reload Parquet.
        reloaded_parquet = pd.read_parquet(parquet_path)
        parquet_check = validate_loaded(card_prices, reloaded_parquet)
        print("Parquet validation:", parquet_check)
    except Exception as error:
        print("Parquet read failed:", error)

CSV validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}
Parquet validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}


## Utilities

In [7]:
util_csv_path = RAW / f"util_{stamp}.csv"
util_parquet_path = PROC / f"util_{stamp}.parquet"

# Write and read CSV.
write_df(card_prices, util_csv_path)
util_csv = read_df(util_csv_path)
print("Util CSV shape:", util_csv.shape)
print("Util CSV validation:", validate_loaded(card_prices, util_csv))

try:
    # Write and read Parquet.
    write_df(card_prices, util_parquet_path)
    util_parquet = read_df(util_parquet_path)
    print("Util Parquet shape:", util_parquet.shape)
    print("Util Parquet validation:", validate_loaded(card_prices, util_parquet))
except RuntimeError as error:
    print("Skipping Parquet util demo:", error)

Util CSV shape: (80, 4)
Util CSV validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}
Util Parquet shape: (80, 4)
Util Parquet validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}


## Documentation

- The paths I use, `DATA_DIR_RAW` and `DATA_DIR_PROCESSED` in `.env`.
- My generated CSV goes into `data/raw/`, and the parquet goes in `data/processed/`.
- My validation code checks shape, columns, date dtype, and numeric `market_price`.
- `.env` is gitignored. I keep `.env.example` in the repo.
- I added the required README Data Storage section in `project/README.md`.

Assumptions and risks:
- Parquet needs `pyarrow` or `fastparquet`.
- CSV can change dtypes if I skip `parse_dates` when running code.